In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
df = pd.read_csv("imdb_clean.csv")

print(df.shape)
df.head()

(994, 15)


,title,director,release_year,runtime,genre,rating,metascore,gross,rating_rank,gross_rank,primary_genre,era,rating_tier,gross_tier,runtime_tier
0,The Shawshank Redemption,Frank Darabont,1994,142.0,Drama,9.3,82,28.34000,1,380,Drama,Blockbuster Era (1990-2009),Elite (9.0+),High,Long (120-150)
1,The Godfather,Francis Ford Coppola,1972,175.0,"Crime, Drama",9.2,100,134.97000,2,141,Crime,New Hollywood (1970-89),Elite (9.0+),Blockbuster,Epic (150+)
2,The Dark Knight,Christopher Nolan,2008,152.0,"Action, Crime, Drama",9.0,84,145.50625,3,10,Action,Blockbuster Era (1990-2009),Elite (9.0+),Blockbuster,Epic (150+)
3,Schindler's List,Steven Spielberg,1993,190.5,"Biography, Drama, History",9.0,95,96.90000,3,192,Biography,Blockbuster Era (1990-2009),Elite (9.0+),Blockbuster,Epic (150+)
4,12 Angry Men,Sidney Lumet,1957,96.0,"Crime, Drama",9.0,97,4.36000,3,577,Crime,Golden Age (1950-69),Elite (9.0+),Low,Standard (90-120)


In [3]:
features = ['primary_genre', 'director', 'rating_tier', 'era']

df['combined_features'] = df[features].astype(str).agg(' '.join, axis=1)

df[['title', 'combined_features']].head()

,title,combined_features
0,The Shawshank Redemption,Drama Frank Darabont Elite (9.0+) Blockbuster ...
1,The Godfather,Crime Francis Ford Coppola Elite (9.0+) New Ho...
2,The Dark Knight,Action Christopher Nolan Elite (9.0+) Blockbus...
3,Schindler's List,Biography Steven Spielberg Elite (9.0+) Blockb...
4,12 Angry Men,Crime Sidney Lumet Elite (9.0+) Golden Age (19...


In [4]:
tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(df['combined_features'])

print(tfidf_matrix.shape)

(994, 929)


In [5]:
# تحويل أسماء الأفلام لindex عشان نقدر نجيب مكان الفيلم في المصفوفة
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

In [8]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(cosine_sim.shape)

(994, 994)


I definitely recommend_movies (title, cosine_sim=cosine_sim):
    
    # If the movie does not exist >> "The movie was not found in the dataset"
    #take index of the movie
    #We calculate the similarity between the movie and other movies
    # We rank from highest to lowest (most similar)
    #We take the best 10 (other than the same)
    # We answer the names of the films

In [9]:
def recommend_movies(title, cosine_sim=cosine_sim):
    
        if title not in indices:
          return "Movie not found in dataset"
    
        idx = indices[title]
    
        sim_scores = list(enumerate(cosine_sim[idx]))
    
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
  
        sim_scores = sim_scores[1:11]
    
        movie_indices = [i[0] for i in sim_scores]
    
        return df['title'].iloc[movie_indices]

In [10]:
recommend_movies("The Dark Knight")

148                                    Batman Begins
42                                      The Prestige
599                                          Dunkirk
66                             The Dark Knight Rises
70                                           Memento
9                                          Inception
17                                      Interstellar
5      The Lord of the Rings: The Return of the King
3                                   Schindler's List
0                           The Shawshank Redemption
Name: title, dtype: object

In [11]:
print(recommend_movies("The Godfather"))

6            The Godfather Part II
661               The Conversation
63                  Apocalypse Now
149                    Taxi Driver
99              A Clockwork Orange
779                    Blue Velvet
104    Once Upon a Time in America
688          The Day of the Jackal
636               The Untouchables
94                        Scarface
Name: title, dtype: object
